In [1]:
!pip install fastapi pydantic httpx pandas

In [2]:
import datetime
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient
raw_dataset = [
    {"Timestamp": "08:00", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 18, "Speed_kmh": 31, "Status": "ON_ROUTE"},
    {"Timestamp": "08:01", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 22, "Speed_kmh": 28, "Status": "ON_ROUTE"},
    {"Timestamp": "08:02", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 21, "Speed_kmh": 29, "Status": "ON_ROUTE"},
    {"Timestamp": "08:03", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 25, "Speed_kmh": 27, "Status": "ON_ROUTE"},
    {"Timestamp": "08:04", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 24, "Speed_kmh": 0, "Status": "STOPPED"},
    {"Timestamp": "08:05", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 26, "Speed_kmh": 30, "Status": "ON_ROUTE"},
    {"Timestamp": "08:06", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 23, "Speed_kmh": 32, "Status": "ON_ROUTE"},
    {"Timestamp": "08:07", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 28, "Speed_kmh": 26, "Status": "ON_ROUTE"},
    {"Timestamp": "08:08", "Route": "AITU-Campus–Residence", "Bus": "B01", "Passengers": 27, "Speed_kmh": 25, "Status": "ON_ROUTE"},
    {"Timestamp": "08:09", "Route": "AITU-Campus–Residence", "Bus": "B02", "Passengers": 30, "Speed_kmh": 24, "Status": "ON_ROUTE"}
]

In [3]:
!pip install httpx2

In [10]:
def parse_and_validate(event: dict) -> dict:
    if event["Passengers"] < 0:
        raise ValueError("number of passengers can not be negative")
    if not (0 <= event["Speed_kmh"] <= 120):
        raise ValueError("Speed out of bounds")
    if event["Status"] not in ["ON_ROUTE", "STOPPED"]:
        raise ValueError("Invalid status") 
    parsed_event = event.copy()
    parsed_event["Timestamp"] = datetime.datetime.strptime(event["Timestamp"], "%H:%M").time()
    return parsed_event
def event_stream_generator(data_source):
    """Yields events one by one without loading everything into memory."""
    for event in data_source:
        try:
            yield parse_and_validate(event)
        except ValueError as e:
            print(f"Skipped invalid event: {e}")
processed_stream = list(event_stream_generator(raw_dataset))
print(f"Events: {len(processed_stream)}")

Events: 10


In [6]:
df = pd.DataFrame(processed_stream)
avg_pax = df['Passengers'].mean()
max_pax = df['Passengers'].max()
stopped_count = df[df['Status'] == 'STOPPED'].shape[0]
busiest_row = df.loc[df['Passengers'].idxmax()]
busiest_minute = busiest_row['Timestamp'].strftime("%H:%M")
busiest_bus = busiest_row['Bus']
print(f"Average Passengers: {avg_pax}")
print(f"Maximum Passengers: {max_pax}")
print(f"STOPPED Events: {stopped_count}")
print(f"Busiest Minute/Bus: {busiest_minute} on {busiest_bus}")

Average Passengers: 24.4
Maximum Passengers: 30
STOPPED Events: 1
Busiest Minute/Bus: 08:09 on B02


In [7]:
app = FastAPI(title="AITU Campus Shuttle API")
class ShuttleEvent(BaseModel):
    Timestamp: str
    Route: str
    Bus: str
    Passengers: int = Field(ge=0)
    Speed_kmh: int = Field(ge=0, le=120)
    Status: str
def calculate_occupancy(passengers: int) -> str:
    if 0 <= passengers <= 10: return "LOW"
    elif 11 <= passengers <= 20: return "MEDIUM"
    elif 21 <= passengers <= 30: return "HIGH"
    return "OVER_CAPACITY"
@app.post("/events")
async def process_event(event: ShuttleEvent):
    if event.Status not in ["ON_ROUTE", "STOPPED"]:
        raise HTTPException(status_code=422, detail="Invalid status")
    return {
        "status": "accepted",
        "occupancy_category": calculate_occupancy(event.Passengers)
    }

In [8]:
!pip install nest-asyncio

In [9]:
import httpx
async def run_tests_async():
    transport = httpx.ASGITransport(app=app)    
    async with httpx.AsyncClient(transport=transport, base_url="http://testserver") as client:
        r1 = await client.post("/events", json={
            "Timestamp": "08:00", "Route": "AITU", "Bus": "B01", 
            "Passengers": 18, "Speed_kmh": 31, "Status": "ON_ROUTE"
        })
        assert r1.status_code == 200
        assert r1.json()["occupancy_category"] == "MEDIUM"
        print("Test 1 Passed")
        r2 = await client.post("/events", json={
            "Timestamp": "08:00", "Route": "AITU", "Bus": "B01", 
            "Passengers": -5, "Speed_kmh": 31, "Status": "ON_ROUTE"
        })
        assert r2.status_code == 422
        print("Test 2 Passed")
        r3 = await client.post("/events", json={
            "Timestamp": "08:00", "Route": "AITU", "Bus": "B01", 
            "Passengers": 10, "Speed_kmh": 150, "Status": "ON_ROUTE"
        })
        assert r3.status_code == 422
        print("Test 3 Passed")
        print("All API tests passed successfully inside the notebook!")
await run_tests_async()

Test 1 Passed
Test 2 Passed
Test 3 Passed
All API tests passed successfully inside the notebook!
